# LBNL Building 59 -- data-quality exploration

Duplicate-timestamp and missing-value checks across the whole 3-year dataset (not just the previously-committed window), to support choosing the cleanest analysis window. Logic lives in `exploration.py`/`loader.py` -- this notebook just calls those functions and displays the resulting `DataFrame`s, which is far more readable than the same tables printed to a terminal.

Run cells top to bottom. `pip install matplotlib` first if the heatmap cells at the bottom error on `.style.background_gradient` (it needs matplotlib for the colormap).

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd


def _find_project_root(start: Path) -> Path:
    """Walk upward from the kernel's cwd (whatever it happens to be --
    notebook-own-folder in VS Code, server-start-dir in classic Jupyter)
    until we find the repo root, identified by requirements.txt."""
    for p in [start, *start.parents]:
        if (p / 'requirements.txt').exists():
            return p
    raise RuntimeError("Could not locate project root (looked for requirements.txt)")


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.lbnl_b59.loader import LBNLLoader, WINDOW_START, WINDOW_END
from src.data.lbnl_b59.preprocessing import clip_window, remove_backward_jump, resample_grid, pad_to_window, build_combined_frame
from src.data.lbnl_b59.exploration import (
    cols_by_file_all,
    duplicate_report, missing_value_report,
    monthly_duplicate_report, monthly_missing_value_report,
    compare_periods, period_summary,
    FULL_SPAN_START, FULL_SPAN_END,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

loader = LBNLLoader(os.path.join('data', 'lbnl_b59', 'raw'))
cols_by_file = cols_by_file_all(loader)
raw_by_file = loader.load_all_raw(cols_by_file)
print(f"{len(cols_by_file)} raw CSV files found")

# Kept files/columns only -- EXCLUDED_CSV_FILES applied, and ttl-matched via
# load_csv_column_attachments() -- i.e. exactly what the rest of the pipeline
# actually sees, for the "committed window check" section near the bottom.
entities_relations = loader.load_entities_and_relations()
entities, relations = entities_relations['entities'], entities_relations['relations']
attachments_result = loader.load_csv_column_attachments(entities, relations)
attachments, unmatched_cols = attachments_result['attachments'], attachments_result['unmatched']
cols_by_file_kept = loader.cols_by_file_from_attachments(attachments)
raw_by_file_kept = {f: raw_by_file[f] for f in cols_by_file_kept}
print(f"{len(cols_by_file_kept)} kept files, {sum(len(c) for c in cols_by_file_kept.values())} attached columns, "
      f"{len(unmatched_cols)} unmatched columns left out")

27 raw CSV files found
19 kept files, 310 attached columns, 15 unmatched columns left out


## 1. Duplicate-timestamp check (whole dataset, every raw file)

`dup_timestamp_rate` = fraction of this file's *distinct* timestamps that have more than one row (not fraction of raw rows -- see exploration.py's comment on why that denominator was wrong). `dup_mismatch_rate` = of the duplicated timestamps, what fraction of duplicates timestamp pairs actually have different values. We can't simply drop them because we consider them to be actually different measurements within same timstamp bucket, as we can see that dup_mismatch_rate shows that duplicate timestamps have varying corresponding values. Here it can also be seen that 'uft_hw_valve.csv' also contains a backword jump where after inspection it looks like they have appended the exact copy of the 2020 data to the end of the file by accident and therefore needs to be removed.

In [2]:
dup_df = duplicate_report(raw_by_file)
dup_df.sort_values('dup_timestamp_rate', ascending=False)

,file,n_rows,n_unique_timestamps,n_backward_jumps,n_dup_rows,n_dup_timestamps,dup_timestamp_rate,n_dup_pairs_checked,n_dup_pairs_mismatched,dup_mismatch_rate
20,uft_hw_valve.csv,2069275,1476006,1,593269,525601,0.356097,593269,73860,0.124497
23,zone_temp_exterior.csv,1510464,1476008,0,34456,28638,0.019402,34456,33735,0.979075
19,uft_fan_spd.csv,1509821,1476337,0,33484,28080,0.019020,33484,25598,0.764485
22,zone_co2.csv,564041,552278,0,11763,8295,0.015020,11763,11763,1.000000
21,wifi.csv,100760,100748,0,12,12,0.000119,12,10,0.833333
11,rtu_oa_t.csv,1381338,1381287,0,51,48,0.000035,51,51,1.000000
13,rtu_ra_t.csv,1381338,1381287,0,51,48,0.000035,51,51,1.000000
8,rtu_ma_t.csv,1381338,1381287,0,51,48,0.000035,51,49,0.960784
10,rtu_oa_fr.csv,1382783,1382732,0,51,48,0.000035,51,45,0.882353
6,rtu_econ_sp.csv,1382783,1382732,0,51,48,0.000035,51,18,0.352941


Showcases that a single timestamp can have more than 1 duplicate per file (in this case 'uft_hw_valve.csv'). This together with the previous finding of varying values per timestamp, leads to the need to average over these duplicates instead of simply dropping them.

In [3]:
valves_df = pd.read_csv(os.path.join(loader.csv_dir, 'uft_hw_valve.csv'))
print(f"Minimum valve value: {valves_df.describe().loc['min'].min()}\n")
print(f"Maximum valve value: {valves_df.describe().loc['max'].max()}\n")

valves_df['date'] = pd.to_datetime(valves_df['date'])
valves_df = valves_df.set_index('date').sort_index()

dupes_df = valves_df[valves_df.index.duplicated(keep=False)]

print(f"The 15 most occurring timestamps in uft_hw_valve.csv:\n{dupes_df.index.value_counts().sort_values(ascending=False).head(15)}")

Minimum valve value: -17.919060611029675

Maximum valve value: 115.411746813143

The 15 most occurring timestamps in uft_hw_valve.csv:
date
2020-03-18 15:49:00    18
2020-07-11 01:26:00    16
2020-07-13 03:45:00    16
2020-12-16 23:55:00    16
2020-10-07 20:51:00    14
2020-10-10 02:07:00    14
2020-02-22 02:37:00    10
2020-02-23 21:03:00    10
2020-10-07 20:52:00    10
2020-10-10 02:08:00    10
2020-01-08 17:06:00     8
2020-01-08 17:19:00     8
2020-01-08 17:21:00     8
2020-01-08 17:54:00     8
2020-01-08 17:56:00     8
Name: count, dtype: int64


Looking at one of the found duplicate timestamps below showcases that these duplicate timestamps are likely valid measurements binned into the same timestamp. For 2020-01-08 17:06:00 one of the columns, zone_055_hw_valve, showcases that in the first 4 rows its value steadily decreases while the remaining 4 are from the entirely duplicated 2020 year.

In [4]:
dupes_df.loc[pd.Timestamp(2020, 1, 8, 17, 6, 0)]

,zone_016_hw_valve,zone_017_hw_valve,zone_018_hw_valve,zone_021_hw_valve,zone_022_hw_valve,zone_023_hw_valve,zone_024_hw_valve,zone_025_hw_valve,zone_026_hw_valve,zone_027_hw_valve,zone_028_hw_valve,zone_035_hw_valve,zone_036_hw_valve,zone_037_hw_valve,zone_038_hw_valve,zone_039_hw_valve,zone_040_hw_valve,zone_041_hw_valve,zone_042_hw_valve,zone_043_hw_valve,zone_044_hw_valve,zone_045_hw_valve,zone_046_hw_valve,zone_047_hw_valve,zone_048_hw_valve,zone_051_hw_valve,zone_052_hw_valve,zone_053_hw_valve,zone_054_hw_valve,zone_055_hw_valve,zone_056_hw_valve,zone_057_hw_valve,zone_058_hw_valve,zone_061_hw_valve,zone_063_hw_valve,zone_064_hw_valve,zone_065_hw_valve,zone_066_hw_valve,zone_067_hw_valve,zone_068_hw_valve,zone_069_hw_valve,zone_070_hw_valve,zone_071_hw_valve,zone_072_hw_valve
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2020-01-08 17:06:00,100.0,100.0,100.0,100.0,50.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,96.0,100.0,0.0,0.0,0.0,100.0,0.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0
2020-01-08 17:06:00,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,95.5,100.0,0.0,0.0,0.0,100.0,0.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0
2020-01-08 17:06:00,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,95.0,100.0,0.0,0.0,0.0,100.0,0.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0
2020-01-08 17:06:00,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,94.5,100.0,0.0,0.0,0.0,100.0,0.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0
2020-01-08 17:06:00,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,95.0,100.0,0.0,0.0,0.0,100.0,0.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0
2020-01-08 17:06:00,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,95.5,100.0,0.0,0.0,0.0,100.0,0.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0
2020-01-08 17:06:00,100.0,100.0,100.0,100.0,50.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,96.0,100.0,0.0,0.0,0.0,100.0,0.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0
2020-01-08 17:06:00,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,94.5,100.0,0.0,0.0,0.0,100.0,0.0,100.0,100.0,100.0,0.0,100.0,100.0,100.0,100.0


### Duplicate check after truncation + resampling

Reruns the duplicate-timestamp check above, but on `clip_window(remove_backward_jump(...))`'s output (after the backward-jump chunk is dropped) instead of the raw CSV -- for `uft_hw_valve.csv`, `dup_timestamp_rate_post_truncation` should drop from 35.6% down to roughly the same small rate as the other files, confirming the truncation fix actually worked rather than assuming it. `n_dup_timestamps_resampled` checks `resample_grid()`+`pad_to_window()`'s final (resampled) output: this is guaranteed to be 0 for every file by how `resample()` works (always one row per bin), so it's shown here to verify that directly rather than take it on faith.

In [5]:
post_check_rows = []
for fname, cols in cols_by_file.items():
    raw = clip_window(remove_backward_jump(raw_by_file[fname]), FULL_SPAN_START, FULL_SPAN_END)
    resampled = pad_to_window(resample_grid(raw), FULL_SPAN_START, FULL_SPAN_END)

    n_unique = raw.index.nunique()
    n_dup_timestamps = raw.index[raw.index.duplicated(keep=False)].nunique()

    post_check_rows.append({
        "file": fname,
        "dup_timestamp_rate_post_truncation": n_dup_timestamps / n_unique if n_unique else 0.0,
        "n_dup_timestamps_resampled": resampled.index.duplicated().sum(),
    })

pd.DataFrame(post_check_rows).sort_values('dup_timestamp_rate_post_truncation', ascending=False)

,file,dup_timestamp_rate_post_truncation,n_dup_timestamps_resampled
23,zone_temp_exterior.csv,0.019402,0
20,uft_hw_valve.csv,0.019163,0
19,uft_fan_spd.csv,0.019020,0
22,zone_co2.csv,0.015020,0
21,wifi.csv,0.000119,0
11,rtu_oa_t.csv,0.000035,0
13,rtu_ra_t.csv,0.000035,0
8,rtu_ma_t.csv,0.000035,0
10,rtu_oa_fr.csv,0.000035,0
6,rtu_econ_sp.csv,0.000035,0


## 2. Monthly duplicate heatmap

File x month `dup_timestamp_rate` (fraction of that month's distinct timestamps duplicated). `uft_hw_valve.csv` should read a flat 100% for every month of 2020 -- confirmed separately (not just by matching timestamps) that this is an exact whole-year copy-paste: every one of its 527,040 unique 2020 timestamps matches, across all 44 columns, between the pre-jump and appended copies.

In [6]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors

monthly_dup_df = monthly_duplicate_report(raw_by_file)
dup_pivot = monthly_dup_df.pivot_table(index='file', columns='month', values='dup_timestamp_rate')

# log normalization such that lower percentages also recieve a color to distinguish them from 0
norm = colors.SymLogNorm(
    linthresh=0.001,
    vmin=0,
    vmax=100
)

cmap = plt.get_cmap('Reds')

def color_cells(val):
    if pd.isna(val):
        return ''
    return f'background-color: {colors.to_hex(cmap(norm(val)))}'

(dup_pivot * 100).style.map(color_cells).format('{:.6f}')

month,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12
file,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ashp_cw.csv,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.000000,0.000000,0.000000,0.000000,0.000000
ashp_hw.csv,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.011201,0.000000,0.000000,0.000000,0.000000,0.000000
ashp_meter.csv,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.000000,0.000000,0.000000,0.000000,0.000000
ele.csv,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
hp_hws_temp.csv,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.007184,0.002240,0.000000,0.000000,nan,nan,nan,nan,nan,nan,nan
occ.csv,nan,nan,nan,nan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
rtu_econ_sp.csv,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003684,0.000000,0.000000,0.009259,0.000000,0.000000,0.000000,0.000000,0.016204,0.008961,0.000000,0.000000,0.000000,0.014368,0.002240,0.013889,0.000000,0.000000,0.004755,0.013457,0.000000,0.020161,0.000000,0.004736
rtu_fan_spd.csv,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
rtu_ma_t.csv,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003692,0.000000,0.000000,0.009259,0.000000,0.000000,0.000000,0.000000,0.016204,0.008961,0.000000,0.000000,0.000000,0.014368,0.002240,0.013889,0.000000,0.000000,0.004758,0.013457,0.000000,0.020161,0.000000,0.004736


## 3. Missing-value check (whole dataset, every column of every raw file)

One row per (file, column). `native_missing_rate` is at the file's own sampling rate; `resampled_missing_rate` is after resampling to the pipeline's common 15-min grid (mean) -- some files' missingness is just scattered sub-15-min dropouts that resampling absorbs entirely, so the two columns can differ a lot. `longest_gap_days` is the longest contiguous gap that survives resampling. Note: this does does also include the missingness due to the absence during certain periods, this will be accounted for later on for the selected window. This means that for the files absent during longer periods, resampling won't solve much of the missingness becausse these intervals are too large, containing no data in the 15 minute intervals. Additionally, for sensors being absent during certain periods that are not at the edge of the time window, resampling can also increase the ratio of missing values. This is during these periods resampling also creates new timestamps on the fixed grid without there being any values, this results in more missing values being recorded. An example is showed later on. Here the report should primarily be used to showcase that there are various columns with missing values present also within periods it should be active.

In [7]:
miss_df = missing_value_report(raw_by_file, cols_by_file)
miss_df.sort_values('native_missing_rate', ascending=False).head(50)

,file,column,native_missing_rate,resampled_missing_rate_unpadded,resampled_missing_rate,longest_gap_days
124,uft_fan_spd.csv,zone_070_fan_spd,0.236221,0.289562,0.289562,203.364583
234,zone_temp_exterior.csv,zone_070_temp,0.235887,0.289552,0.289552,203.364583
168,uft_hw_valve.csv,zone_070_hw_valve,0.235649,0.289229,0.289229,203.364583
333,zone_temp_sp_h.csv,zone_070_heating_sp,0.221983,0.242796,0.420801,257.416667
292,zone_temp_sp_c.csv,zone_070_cooling_sp,0.221976,0.242796,0.420801,257.416667
243,zone_temp_interior.csv,cerc_templogger_15,0.168784,0.172671,0.211975,52.020833
81,uft_fan_spd.csv,zone_022_fan_spd,0.087603,0.147221,0.147221,20.541667
131,uft_hw_valve.csv,zone_022_hw_valve,0.085489,0.145461,0.145461,20.541667
191,zone_temp_exterior.csv,zone_022_temp,0.083953,0.143986,0.143986,20.541667
240,zone_temp_interior.csv,cerc_templogger_12,0.081931,0.086211,0.129622,58.062500


This shows that some sensors natively don't have missing values during their up-time

In [8]:
miss_df.sort_values('native_missing_rate', ascending=False).tail(50)

,file,column,native_missing_rate,resampled_missing_rate_unpadded,resampled_missing_rate,longest_gap_days
183,zone_co2.csv,zone_062_co2,0.0,0.232837,0.649737,595.062500
184,zone_co2.csv,zone_068_co2,0.0,0.232837,0.649737,595.062500
43,rtu_plenum_p.csv,rtu_001_fltrd_gnd_lvl_plenum_press_tn,0.0,0.000000,0.000000,0.000000
24,rtu_fan_spd.csv,rtu_002_rf_vfd_spd_fbk_tn,0.0,0.000000,0.000000,0.000000
50,rtu_plenum_p.csv,rtu_004_fltrd_lvl2_plenum_press_tn,0.0,0.000000,0.000000,0.000000
41,rtu_oa_t.csv,rtu_003_oa_temp,0.0,0.000000,0.000000,0.000000
42,rtu_oa_t.csv,rtu_004_oa_temp,0.0,0.000000,0.000000,0.000000
54,rtu_ra_t.csv,rtu_004_ra_temp,0.0,0.000000,0.000000,0.000000
12,hp_hws_temp.csv,hp_hws_temp,0.0,0.145393,0.334662,242.510417
178,zone_co2.csv,zone_040_co2,0.0,0.232837,0.649737,595.062500


This shows that resampling and padding (reindexing to full timewindow) can both inflate the missingness, the main example are the wifi sensors where there are 2 separate periods where the sensors are up, natively in between these two periods there are no timestamps but due to resampling there are timestamps with nan values on the fixed grid. This is why we need to exclude files with overlapping downtime during the chosen time window. Also means choosing timewindow is important

In [9]:
miss_df.sort_values('resampled_missing_rate_unpadded', ascending=False).head(50)

,file,column,native_missing_rate,resampled_missing_rate_unpadded,resampled_missing_rate,longest_gap_days
172,wifi.csv,wifi_second_south,0.072886,0.660032,0.703808,770.677083
171,wifi.csv,wifi_first_south,0.072886,0.660032,0.703808,770.677083
173,wifi.csv,wifi_third_south,0.000000,0.606573,0.657233,578.677083
174,wifi.csv,wifi_fourth_south,0.000000,0.606573,0.657233,578.677083
124,uft_fan_spd.csv,zone_070_fan_spd,0.236221,0.289562,0.289562,203.364583
234,zone_temp_exterior.csv,zone_070_temp,0.235887,0.289552,0.289552,203.364583
168,uft_hw_valve.csv,zone_070_hw_valve,0.235649,0.289229,0.289229,203.364583
292,zone_temp_sp_c.csv,zone_070_cooling_sp,0.221976,0.242796,0.420801,257.416667
333,zone_temp_sp_h.csv,zone_070_heating_sp,0.221983,0.242796,0.420801,257.416667
177,zone_co2.csv,zone_033_co2,0.000000,0.232837,0.649737,595.062500


## 4. Candidate window comparison

Per-window summary (duplicates computed first, missing values on top -- see `period_summary()`). Add more entries to `candidate_windows` below to compare others; `start`/`end` can be any string pandas can parse.

In [10]:
candidate_windows = {
    "full 3-year span": (FULL_SPAN_START, FULL_SPAN_END),
    "committed (2020-05-01 to 2020-12-31)": (WINDOW_START, WINDOW_END),
    "2019-02-20 to 2020-04-01": ('2019-02-20', '2020-04-01'),
    "2019-02-20 to 2019-12-31": ('2019-02-20', '2019-12-31'),
}
summary_df = compare_periods(raw_by_file, cols_by_file, candidate_windows)
summary_df

,label,start,end,n_days,any_backward_jump,mean_dup_timestamp_rate,max_dup_timestamp_rate,mean_resampled_missing_rate,max_resampled_missing_rate
0,full 3-year span,2018-01-01,2020-12-31,1095,True,0.015184,0.356097,0.178914,0.861445
1,committed (2020-05-01 to 2020-12-31),2020-05-01,2020-12-31,244,True,0.041122,1.000000,0.013453,1.000000
2,2019-02-20 to 2020-04-01,2019-02-20,2020-04-01,406,True,0.009115,0.224139,0.060814,1.000000
3,2019-02-20 to 2019-12-31,2019-02-20,2019-12-31,314,False,0.000009,0.000033,0.062343,1.000000


## 5. Monthly missing-value heatmap

File x month, mean `resampled_missing_rate` across that file's columns, colored so clean (white) vs. dirty (dark red) months jump out -- this is the main view for spotting which contiguous span is cleanest.

In [11]:
monthly_miss_df = monthly_missing_value_report(raw_by_file, cols_by_file)
miss_pivot = monthly_miss_df.pivot_table(index='file', columns='month', values='resampled_missing_rate')
(miss_pivot * 100).style.background_gradient(cmap='Reds', vmin=0, vmax=100).format('{:.2f}')

month,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12
file,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ashp_cw.csv,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,0.94,0.00,0.00,0.00,0.00
ashp_hw.csv,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,83.06,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
ashp_meter.csv,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,0.94,0.00,0.00,0.00,0.00
ele.csv,0.13,7.60,9.03,0.00,4.50,0.00,5.97,18.19,0.97,0.00,37.35,0.00,10.14,0.00,0.00,0.00,0.00,0.00,6.92,11.68,0.00,31.07,0.00,3.83,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
hp_hws_temp.csv,44.56,13.02,33.37,18.16,40.49,53.85,35.35,49.26,15.83,4.27,7.05,38.27,42.88,8.63,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,95.16,100.00,100.00,100.00,100.00,100.00,100.00,100.00
occ.csv,100.00,100.00,100.00,100.00,68.68,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,27.05,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
rtu_econ_sp.csv,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
rtu_fan_spd.csv,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
rtu_ma_t.csv,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


## 6. Committed window check (kept files, attached columns only)

Reruns the duplicate and missing-value checks above, but scoped to just the committed window (`WINDOW_START`/`WINDOW_END`) and just `cols_by_file_kept`/`raw_by_file_kept` -- the files/columns that actually survive both `EXCLUDED_CSV_FILES` and the ttl-matching in `load_csv_column_attachments()`. This is exactly what the rest of the pipeline will see, rather than the full unfiltered dataset explored above.

In [12]:
kept_dup_df = duplicate_report(raw_by_file_kept, start=WINDOW_START, end=WINDOW_END)
kept_dup_df

,file,n_rows,n_unique_timestamps,n_backward_jumps,n_dup_rows,n_dup_timestamps,dup_timestamp_rate,n_dup_pairs_checked,n_dup_pairs_mismatched,dup_mismatch_rate
12,uft_hw_valve.csv,739500,351361,1,388139,351361,1.000000,388139,36354,0.093662
15,zone_temp_exterior.csv,370301,351361,0,18940,15380,0.043773,18940,18541,0.978933
11,uft_fan_spd.csv,369632,351361,0,18271,15002,0.042697,18271,13717,0.750753
14,zone_co2.csv,362881,351361,0,11520,8117,0.023102,11520,11520,1.000000
13,wifi.csv,70285,70273,0,12,12,0.000171,12,10,0.833333
7,rtu_ra_t.csv,346965,346944,0,21,19,0.000055,21,21,1.000000
3,rtu_ma_t.csv,346965,346944,0,21,19,0.000055,21,19,0.904762
4,rtu_oa_damper.csv,347018,346997,0,21,19,0.000055,21,12,0.571429
5,rtu_oa_fr.csv,347018,346997,0,21,19,0.000055,21,15,0.714286
1,rtu_econ_sp.csv,347018,346997,0,21,19,0.000055,21,9,0.428571


In [13]:
post_check_rows = []
for fname, cols in cols_by_file_kept.items():
    raw = clip_window(remove_backward_jump(raw_by_file_kept[fname]), WINDOW_START, WINDOW_END)
    resampled = pad_to_window(resample_grid(raw), WINDOW_START, WINDOW_END)

    n_unique = raw.index.nunique()
    n_dup_timestamps = raw.index[raw.index.duplicated(keep=False)].nunique()

    post_check_rows.append({
        "file": fname,
        "dup_timestamp_rate_post_truncation": n_dup_timestamps / n_unique if n_unique else 0.0,
        "n_dup_timestamps_resampled": resampled.index.duplicated().sum(),
    })

pd.DataFrame(post_check_rows).sort_values('dup_timestamp_rate_post_truncation', ascending=False)

,file,dup_timestamp_rate_post_truncation,n_dup_timestamps_resampled
15,zone_temp_exterior.csv,0.043773,0
12,uft_hw_valve.csv,0.042839,0
11,uft_fan_spd.csv,0.042697,0
14,zone_co2.csv,0.023102,0
13,wifi.csv,0.000171,0
7,rtu_ra_t.csv,0.000055,0
3,rtu_ma_t.csv,0.000055,0
4,rtu_oa_damper.csv,0.000055,0
5,rtu_oa_fr.csv,0.000055,0
1,rtu_econ_sp.csv,0.000055,0


In [14]:
kept_miss_df = missing_value_report(raw_by_file_kept, cols_by_file_kept, start=WINDOW_START, end=WINDOW_END)
kept_miss_df.sort_values('resampled_missing_rate', ascending=False)

,file,column,native_missing_rate,resampled_missing_rate_unpadded,resampled_missing_rate,longest_gap_days
0,ele.csv,mels_S,0.0,0.0,0.0,0.0
1,ele.csv,lig_S,0.0,0.0,0.0,0.0
2,ele.csv,mels_N,0.0,0.0,0.0,0.0
3,ele.csv,hvac_N,0.0,0.0,0.0,0.0
4,ele.csv,hvac_S,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...
305,zone_temp_sp_h.csv,zone_066_heating_sp,0.0,0.0,0.0,0.0
306,zone_temp_sp_h.csv,zone_067_heating_sp,0.0,0.0,0.0,0.0
307,zone_temp_sp_h.csv,zone_069_heating_sp,0.0,0.0,0.0,0.0
308,zone_temp_sp_h.csv,zone_070_heating_sp,0.0,0.0,0.0,0.0


In [15]:
combined_df = build_combined_frame(raw_by_file_kept, cols_by_file_kept, WINDOW_START, WINDOW_END)
print(f"combined shape: {combined_df.shape}")
print(f"total NaN cells: {combined_df.isna().sum().sum()}")
print(f"rows with >=1 NaN: {combined_df.isna().any(axis=1).sum()} / {len(combined_df)}")

combined shape: (23425, 310)
total NaN cells: 0
rows with >=1 NaN: 0 / 23425
